# 🎬 MPT Agent System — Gemini Colab Runner

**Runtime: Standard (CPU only)** — No GPU needed.

Pipeline: **TrendScout → ScriptWriter (Gemini) → VideoProducer (MPT stock footage) → SEO (Gemini) → Publisher**

---

## ✅ Step 1 — Clone repos and install dependencies

In [ ]:
# 1a. Clone MoneyPrinterTurbo
import os
if not os.path.exists('/content/MoneyPrinterTurbo'):
    !git clone --depth 1 https://github.com/harry0703/MoneyPrinterTurbo.git /content/MoneyPrinterTurbo
else:
    !cd /content/MoneyPrinterTurbo && git pull --quiet
print('MPT cloned.')

# 1b. Install MPT dependencies from its own requirements.txt
!pip install -q -r /content/MoneyPrinterTurbo/requirements.txt 2>&1 | tail -5

# 1c. Install extra packages needed by MPT internals on Colab
!pip install -q loguru edge-tts imageio-ffmpeg moviepy==1.0.3 2>&1 | tail -3

# 1d. Clone agent system
if not os.path.exists('/content/mpt-agent-system'):
    !git clone --depth 1 https://github.com/Bhushan-Khachane/mpt-agent-system.git /content/mpt-agent-system
else:
    !cd /content/mpt-agent-system && git pull --quiet

# 1e. Install agent system dependencies
!pip install -q feedparser pytrends requests google-api-python-client google-auth-httplib2 google-auth-oauthlib 2>&1 | tail -3

print('\n✅ All dependencies installed.')

## 🔑 Step 2 — Paste your API keys

- **Gemini (free):** https://aistudio.google.com/app/apikey
- **Pexels (free):** https://www.pexels.com/api/
- Pixabay is optional but recommended as fallback.

In [ ]:
import os, sys, pathlib

# ════════════════════════════════════════
# EDIT THESE VALUES
GEMINI_API_KEY  = 'AIza...'           # <-- your real Gemini key
PEXELS_API_KEY  = 'YOUR_PEXELS_KEY'   # <-- your Pexels key (required for stock video)
PIXABAY_API_KEY = ''                  # <-- optional
# ════════════════════════════════════════

# Validate keys
_bad = {'AIza...', 'YOUR_GEMINI_API_KEY', '', None}
if GEMINI_API_KEY in _bad:
    raise ValueError('ERROR: Paste your real Gemini key above. Get it free at https://aistudio.google.com/app/apikey')
if PEXELS_API_KEY in {'YOUR_PEXELS_KEY', '', None}:
    raise ValueError('ERROR: Paste your Pexels API key above. Get it free at https://www.pexels.com/api/')

# Set env vars
os.environ['GEMINI_API_KEY']  = GEMINI_API_KEY
os.environ['GEMINI_MODEL']    = 'gemini-2.0-flash'
os.environ['PEXELS_API_KEY']  = PEXELS_API_KEY
os.environ['PIXABAY_API_KEY'] = PIXABAY_API_KEY
os.environ['MPT_DIR']         = '/content/MoneyPrinterTurbo'
os.environ['YT_CHANNEL_AI']    = 'UCxxxxxxxx'  # optional — for publisher step
os.environ['YT_CHANNEL_CLEAN'] = 'UCxxxxxxxx'
os.environ['YT_CHANNEL_MONEY'] = 'UCxxxxxxxx'

# Add agent system to Python path
AGENT_ROOT = '/content/mpt-agent-system'
if AGENT_ROOT not in sys.path:
    sys.path.insert(0, AGENT_ROOT)

# Create required directories and empty data files
for d in ['data', 'videos', 'logs', 'credentials']:
    pathlib.Path(f'{AGENT_ROOT}/{d}').mkdir(parents=True, exist_ok=True)
for fp, default in [
    (f'{AGENT_ROOT}/data/topics_queue.json', '[]'),
    (f'{AGENT_ROOT}/data/seen_topics.json',  '[]'),
]:
    p = pathlib.Path(fp)
    if not p.exists():
        p.write_text(default)

print(f'✅ Config OK | Model: {os.environ["GEMINI_MODEL"]} | Key: {GEMINI_API_KEY[:12]}...')

## 📈 Step 3 — TrendScout (discover topics)

In [ ]:
# NOTE: sys.path is already set in Step 2. Just import and run.
from agents.trend_scout import run_trend_scout
import json

topics = run_trend_scout()
print(f'\n📋 {len(topics)} topics queued. Top 5:')
for t in topics[:5]:
    print(f"  [{t['niche']:12s}] {t['title'][:55]} (score={t.get('score_val',0):.1f})")

## ✍️ Step 4 — ScriptWriter (Gemini API)
> Each script is ~50-80 words, niche-specific, generated by Gemini 2.0 Flash.

In [ ]:
from agents.script_writer import run_script_writer
import json, pathlib

n = run_script_writer(batch_size=6)
print(f'\n✅ {n} scripts written')

queue    = json.loads(pathlib.Path('/content/mpt-agent-system/data/topics_queue.json').read_text())
scripted = [t for t in queue if t.get('status') == 'scripted']
if scripted:
    t0 = scripted[0]
    print(f"\n--- Sample ({t0['niche']}) ---")
    print(t0['script'])

## 🎬 Step 5 — VideoProducer (MPT stock footage)
> Calls MPT CLI directly (`python cli.py ...`). Uses Pexels stock videos + edge-tts voiceover.
> Each video takes **2-5 minutes**. Batch of 3 = ~10-15 min.

In [ ]:
from agents.video_producer import run_video_producer
run_video_producer(batch_size=2)  # Start with 2 to test — increase later

## 🏷️ Step 6 — SEO Agent (Gemini API)

In [ ]:
from agents.seo_agent import run_seo_agent
run_seo_agent(batch_size=6)

## 📤 Step 7 — Publisher (YouTube)
> **With OAuth token:** Upload `credentials/youtube_<niche>_token.json` to automatically publish.
> **Without token:** Videos + SEO are saved to `data/manual_upload_queue.json` — upload manually.

In [ ]:
from agents.publisher import run_publisher
run_publisher(batch_size=3)

## 📋 Check pipeline status

In [ ]:
import json, pathlib
queue = json.loads(pathlib.Path('/content/mpt-agent-system/data/topics_queue.json').read_text())
from collections import Counter
counts = Counter(t.get('status','unknown') for t in queue)
print('Pipeline status:')
for status, n in sorted(counts.items()):
    print(f'  {status:30s}: {n}')

# Show videos ready for manual upload
manual = pathlib.Path('/content/mpt-agent-system/data/manual_upload_queue.json')
if manual.exists():
    items = json.loads(manual.read_text())
    if items:
        print(f'\n📁 {len(items)} videos ready for manual upload:')
        for v in items:
            print(f"  [{v['niche']}] {v['title'][:50]}")
            print(f"  File: {v['video_path']}")
            print(f"  YT Title: {v['youtube_title']}")
            print()

## ⚡ One-shot: Run full pipeline (Steps 3–7)

In [ ]:
# Make sure you have run Steps 1 and 2 first!
from agents.trend_scout   import run_trend_scout
from agents.script_writer import run_script_writer
from agents.video_producer import run_video_producer
from agents.seo_agent     import run_seo_agent
from agents.publisher     import run_publisher

print('=== TrendScout ==='); run_trend_scout()
print('=== ScriptWriter ==='); run_script_writer(batch_size=6)
print('=== VideoProducer ==='); run_video_producer(batch_size=2)
print('=== SEOAgent ==='); run_seo_agent(batch_size=6)
print('=== Publisher ==='); run_publisher(batch_size=3)
print('=== Done ===')